# Software Engineering Agents

Software engineering (SWE) agents go beyond chat: they plan, use tools (shell, search, tests), and iterate toward a goal such as fixing an issue. This notebook covers architectures, planning styles, failure modes, and a ReAct-style skeleton.


## Learning Objectives

- Contrast chat assistants vs SWE agents
- Design a minimal tool surface
- Compare planning styles
- Detect loops and other failure modes


## 1. From Chat to Agents

| Dimension | Chat | Agent |
|-----------|------|-------|
| Steps | 1–few | Many |
| Tools | Optional | Central |
| State | Conversation | Workspace + memory |
| Success | User likes answer | Tests/CI green |
| Risk | Bad advice | Bad actions |

```mermaid
flowchart LR
  G[Goal] --> P[Plan]
  P --> A[Act: tool]
  A --> O[Observe]
  O --> P
  O --> D[Done / escalate]
```


## 2. Tool Surface Area

Minimum useful tools:
- `read_file` / `write_file` / `apply_patch`
- `grep` / `glob`
- `run_tests` / `run_cmd` (sandboxed)
- `git_status` / `git_diff`

### Pitfalls
Unsandboxed shell is a critical security risk—see Module 22.


In [ ]:
# Demo 1 — Minimal ReAct-style coding agent skeleton
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Callable

@dataclass
class AgentResult:
    done: bool
    output: str

Tool = Callable[[str], str]

@dataclass
class CodingAgent:
    tools: dict[str, Tool]
    max_steps: int = 8
    trace: list[str] = field(default_factory=list)

    def step(self, thought: str, tool: str, arg: str) -> str:
        self.trace.append(f"THOUGHT: {thought}")
        if tool not in self.tools:
            obs = f"unknown tool {tool}"
        else:
            obs = self.tools[tool](arg)
        self.trace.append(f"ACT: {tool}({arg!r}) -> {obs}")
        return obs

def run_demo():
    files = {"app.py": "def add(a,b):\n    return a-b\n"}
    def read_file(p: str) -> str:
        return files.get(p, "NOT FOUND")
    def run_tests(_: str) -> str:
        # buggy until patched externally
        return "FAIL" if "return a-b" in files["app.py"] else "PASS"
    def apply_patch(_: str) -> str:
        files["app.py"] = "def add(a,b):\n    return a+b\n"
        return "patched"
    agent = CodingAgent({"read_file": read_file, "run_tests": run_tests, "apply_patch": apply_patch})
    agent.step("inspect", "read_file", "app.py")
    agent.step("see failure", "run_tests", "")
    agent.step("fix add", "apply_patch", "add")
    agent.step("verify", "run_tests", "")
    return agent.trace

for line in run_demo():
    print(line)


## 3. Planning Styles

| Style | How | When |
|-------|-----|------|
| ReAct | Interleave think/act | Exploratory bugs |
| Plan-then-execute | Full plan first | Well-scoped migrations |
| Hierarchical | Manager + workers | Large tasks |
| Test-driven | Tests as north star | Clear acceptance |

### Intuition
Plans without environment feedback rot quickly; pure reactive loops thrash. Hybrid usually wins.


## 4. Failure Modes & Mitigations

| Failure | Symptom | Mitigation |
|---------|---------|------------|
| Looping | Repeated same tool args | Fingerprint + budget |
| Thrashing | Oscillating patches | Freeze files / bisect |
| Spec drift | Edits tests wrongly | Protect test perms |
| Context bloat | Huge prompts | Summarize + retrieve |
| Hallucinated APIs | Import errors | Typecheck early |


In [ ]:
# Demo 2 — Detect repeated-state loops
from hashlib import sha1

def state_fingerprint(files: dict[str, str], last_tool: str) -> str:
    blob = last_tool + "||" + "||".join(f"{k}:{sha1(v.encode()).hexdigest()[:8]}" for k, v in sorted(files.items()))
    return sha1(blob.encode()).hexdigest()[:12]

seen = set()
files = {"a.py": "x=1\n"}
for i in range(5):
    fp = state_fingerprint(files, "read_file")
    loop = fp in seen
    seen.add(fp)
    print(i, fp, "LOOP" if loop else "ok")


In [ ]:
# Demo 3 — Step / cost budget
from dataclasses import dataclass

@dataclass
class Budget:
    max_steps: int = 10
    max_shell: int = 5
    steps: int = 0
    shell: int = 0

    def allow(self, tool: str) -> bool:
        self.steps += 1
        if self.steps > self.max_steps:
            return False
        if tool == "run_cmd":
            self.shell += 1
            if self.shell > self.max_shell:
                return False
        return True

b = Budget()
print([b.allow("read_file") for _ in range(3)], b.steps)


### Try it yourself — SWE Agents

- Add a `grep` tool to the skeleton and fix a bug in a 3-file toy repo
- Implement loop detection that aborts after 2 repeated fingerprints
- Write a policy: which tools require human approval?


## Interview Prep — Sample Q&A

Practice answering out loud, then compare to the sample answers.


### Q1. How do you stop an SWE agent from looping?

**Sample answer**

Track state fingerprints, cap steps/tool calls, detect repeated failing patches, escalate to humans, and protect tests from unauthorized edits.


## Deep Dive Workshop — 06 Software Engineering Agents

This section expands the notebook into instructor/textbook depth. Work through each subsection: **definition → why it matters → how it works → intuition → pitfalls → when to use**.

```mermaid
flowchart TB
  D[Definition] --> W[Why it matters]
  W --> H[How it works]
  H --> I[Intuition]
  I --> P[Pitfalls]
  P --> U[When to use]
```


### Concept card pack for `06-software-engineering-agents`

| Concept | Definition | Why it matters | Common pitfall |
|---------|------------|----------------|----------------|
| Primary abstraction | Core object this lesson centers on | Anchors design conversations | Vague naming |
| Quality oracle | How you know the system is right | Prevents demo-driven development | Using vibes only |
| Latency budget | Max user-visible wait | Drives architecture | Ignoring TTFT vs e2e |
| Cost unit | $ per successful task | Makes tradeoffs real | Optimizing tokens not outcomes |
| Trust boundary | Where data/control changes hands | Security design | Treating vendors as internal |
| Feedback loop | How production improves the system | Sustainable quality | No path from thumbs-down to evals |

**Intuition:** If you cannot fill this table for your system, you are not ready to choose models or frameworks.


### Pipeline walkthrough (apply to 06-software-engineering-agents)

```
1. Input arrives (user / job / webhook)
2. Normalize + authorize + budget check
3. Gather context (files, RAG, tools, memory)
4. Model / deterministic compute
5. Validate output (schema, policy, tests)
6. Side effects (write, ticket, PR) with authz
7. Observe (metrics, traces, feedback)
8. Learn (eval suite growth, prompt/model revision)
```

**When to compress steps:** tiny internal tools. **When to keep all steps:** multi-tenant or regulated production.


### Coding-models advanced notes

**Fill-in-the-middle formats** differ by vendor; always keep an adapter layer.  
**Repo agents** should treat tests as the north star and protect test files by default.  
**Coding RAG** should combine symbol lookup + BM25 + embeddings; embeddings alone miss identifiers.

| Task | Prefer | Avoid |
|------|--------|-------|
| Ghost text | FIM-capable small/fast model | Giant chat model sync |
| API migration | Agent + tests | Single-shot whole-repo rewrite |
| Explain legacy | Chat + citations | Uncited summaries |


In [ ]:
# Extra demo — diff extraction toy for coding assistants
import re

def extract_fenced_blocks(text: str) -> list[tuple[str, str]]:
    pat = re.compile(r"```(\w+)?\n(.*?)```", re.S)
    return [(m.group(1) or 'txt', m.group(2)) for m in pat.finditer(text)]

sample = '''Here is a fix:\n```python\ndef add(a,b):\n    return a+b\n```\n'''
print(extract_fenced_blocks(sample))


In [ ]:
# Extra demo — simple symbol index for coding RAG
import ast
from collections import defaultdict

def index_symbols(source: str, path: str) -> dict[str, list[str]]:
    tree = ast.parse(source)
    idx = defaultdict(list)
    for n in ast.walk(tree):
        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            idx[n.name].append(f"{path}:{n.lineno}")
    return dict(idx)

print(index_symbols('class Foo:\n  def bar(self):\n    pass\n', 'a.py'))


### Sample interview Q&A — coding models

**Q:** Copilot-quality inline completion is slow. What do you do?  
**A:** Separate completion model from chat model; shrink context to locals + imports; consider speculative decoding / smaller quantized model; measure acceptance rate not just tok/s.

**Q:** How do you evaluate a coding assistant for a monorepo?  
**A:** Private suite: completion acceptance, unit-test pass on generated patches, security scanner findings, and human review on a stratified sample of PR diffs.


In [ ]:
# Extra demo — tool permission matrix
perms = {
    'junior_agent': {'read_file', 'grep', 'run_tests'},
    'senior_agent': {'read_file', 'grep', 'run_tests', 'apply_patch'},
    'human': {'read_file', 'grep', 'run_tests', 'apply_patch', 'git_push'},
}
def allowed(role, tool):
    return tool in perms.get(role, set())
print(allowed('junior_agent','apply_patch'), allowed('senior_agent','apply_patch'))


### Comparison matrix exercise

Fill this for two competing designs in this topic:

| Dimension | Option A | Option B | Winner / why |
|-----------|----------|----------|--------------|
| Latency | | | |
| Cost at 10× scale | | | |
| Quality risk | | | |
| Ops burden | | | |
| Security / privacy | | | |
| Time to MVP | | | |


In [ ]:
# Workshop demo — decision scorecard
from dataclasses import dataclass

@dataclass
class Option:
    name: str
    latency: int  # 1=best .. 5=worst
    cost: int
    quality_risk: int
    ops: int
    security: int

def score(o: Option, weights=None) -> float:
    weights = weights or dict(latency=1, cost=1, quality_risk=2, ops=1, security=2)
    return (
        o.latency*weights['latency'] + o.cost*weights['cost'] +
        o.quality_risk*weights['quality_risk'] + o.ops*weights['ops'] +
        o.security*weights['security']
    )

a = Option('A', 2, 3, 2, 2, 2)
b = Option('B', 3, 1, 3, 4, 2)
print(a.name, score(a), b.name, score(b), '-> prefer', a.name if score(a)<score(b) else b.name)


In [ ]:
# Workshop demo — experiment log (use while studying this notebook)
from dataclasses import dataclass, asdict
import json, time

@dataclass
class Experiment:
    hypothesis: str
    setup: str
    metric: str
    baseline: float | None = None
    treatment: float | None = None
    notes: str = ''
    ts: float = 0.0

    def __post_init__(self):
        if not self.ts:
            self.ts = time.time()

exp = Experiment(
    hypothesis='Technique from this lesson improves the primary metric',
    setup='Describe fixtures / model / dataset version',
    metric='name of metric',
    baseline=0.0,
    treatment=0.0,
)
print(json.dumps(asdict(exp), indent=2))


### ASCII architecture sketch template

```
[ Clients ]
     |
[ Edge / API Gateway ] -- authn/z, rate limit
     |
[ Orchestration ] ------+-- prompts / policies
     |                  +-- eval hooks
     +-- context layer (RAG / tools / memory)
     |
[ Model interface ] ---- local and/or cloud
     |
[ Data plane ] --------- indexes, OLTP, object store
     |
[ Observability ] ------ logs, metrics, traces, feedback
```

Copy into your notes and annotate trust boundaries with `***`.


### Pitfalls clinic (read aloud)

1. **Metric theater** — optimizing a proxy that users don't feel  
2. **Context stuffing** — more tokens ≠ more truth  
3. **Prompt as security** — never the only control  
4. **Hidden coupling** — tools/models/indexes version-drift  
5. **No rollback** — can't revert prompt/model quickly  
6. **Eval contamination** — testing on training-like snippets  
7. **Happy-path demos** — skipping adversarial & empty-retrieve cases  


### Try it yourself — extended set

1. Teach the top 3 ideas from this notebook to a rubber duck in 5 minutes  
2. Write 5 quiz questions (with answers) for a junior engineer  
3. Implement one code demo with a real dependency (API or local model) using env placeholders  
4. Break a naive design on purpose; list the failure mode and the fix  
5. Add two rows to your personal glossary with examples from work  
6. Produce a one-page cheat sheet you could use in an interview  


### Mini case study

**Scenario:** Leadership wants this capability in production in six weeks with two engineers.

**Your job:** Propose an MVP that keeps irreversible risks controlled, names the eval gates, and lists what you explicitly defer.

Deliverable structure:
- MVP user story  
- Non-goals  
- Architecture (6 boxes max)  
- Eval gate table  
- Risk register (top 5)  
- Week-by-week plan  


In [ ]:
# Case study helper — risk register
import pandas as pd

risks = pd.DataFrame([
    {'risk': 'quality_miss', 'likelihood': 3, 'impact': 3, 'mitigation': 'golden evals + canary'},
    {'risk': 'cost_overrun', 'likelihood': 3, 'impact': 2, 'mitigation': 'budgets + cache'},
    {'risk': 'data_leak', 'likelihood': 2, 'impact': 5, 'mitigation': 'ACL + redaction'},
    {'risk': 'prompt_injection', 'likelihood': 4, 'impact': 4, 'mitigation': 'boundaries + allowlists'},
    {'risk': 'ops_pages', 'likelihood': 3, 'impact': 3, 'mitigation': 'runbooks + rollback'},
])
risks['score'] = risks.likelihood * risks.impact
print(risks.sort_values('score', ascending=False).to_string(index=False))


### Interview drill (topic-local)

Use the STAR or design template. Timebox 8 minutes.

**Prompt:** “Walk me through how you would productionize the main idea of this notebook.”

Checklist for a strong answer:
- [ ] Clarifying questions  
- [ ] Constraints & numbers  
- [ ] Diagram  
- [ ] Deep dive on hardest part  
- [ ] Evals  
- [ ] Security  
- [ ] Rollout / rollback  


### Glossary boost

| Term | Expanded meaning |
|------|------------------|
| Canary | Partial traffic to a new variant with automatic rollback |
| Golden set | Versioned labeled examples for regression |
| TTFT | Time to first token — interactive UX driver |
| Packing | Selecting/ordering context under a token budget |
| HITL | Human approval inserted before side effects |
| Idempotency | Safe retries without duplicate side effects |
| Shadow traffic | New system sees traffic but doesn't affect users |
| Circuit breaker | Stop calling a failing dependency temporarily |


In [ ]:
# Self-check quiz (run and answer mentally before printing answers)
QUESTIONS = [
    'What oracle proves success for this topic?',
    'Name one metric that can be gamed and a better alternative.',
    'What is the top security failure mode?',
    'What would you defer in an MVP?',
    'How do you rollback a bad change here?',
]
for i, q in enumerate(QUESTIONS, 1):
    print(f'Q{i}. {q}')
print('\n--- suggested answer hints ---')
HINTS = [
    'executable tests / task success / human rubric',
    'longer answers != better; use task success',
    'trust boundary crossing / injection / ACL',
    'multi-agent, perfect UI, every connector',
    'versioned prompts/models + traffic switch',
]
for h in HINTS:
    print('-', h)


### Further practice roadmap for `06-software-engineering-agents`

| Horizon | Action |
|---------|--------|
| Today | Re-run all code cells; note questions |
| This week | Apply one technique to a real repo/service |
| This month | Add an eval or security test covering this topic |
| Interview ready | Give a 10-minute teach-back with a diagram |


## Lab: End-to-end scenario

Work this scenario in your notes, then implement the smallest possible spike.

### Scenario brief
A team wants to adopt the techniques from this notebook for a **real internal tool** used daily by 200 people. Leadership cares about reliability and auditability more than flashy demos.

### Deliverables
1. One-paragraph problem statement  
2. Success metrics (3) with oracles  
3. Architecture sketch with trust boundaries  
4. Threats / failure modes (5)  
5. Eval plan (offline + online)  
6. 2-week MVP scope and explicit non-goals  

### Review questions
- What happens when context is empty?  
- What happens when the model is down?  
- What happens when a user is malicious?  
- How do you prove a release is safer/better than last week?  


In [ ]:
# Lab helper — MVP scope tracker
from dataclasses import dataclass, field

@dataclass
class MVP:
    must: list[str] = field(default_factory=list)
    should: list[str] = field(default_factory=list)
    defer: list[str] = field(default_factory=list)

    def show(self):
        for label, items in [('MUST', self.must), ('SHOULD', self.should), ('DEFER', self.defer)]:
            print(label)
            for i in items:
                print(' -', i)

mvp = MVP(
    must=['core happy path', 'authn', 'basic eval smoke', 'rollback switch'],
    should=['streaming UX', 'dashboards'],
    defer=['multi-agent', 'perfect personalization', 'every connector'],
)
mvp.show()


## Operator runbook sketch

| Symptom | Likely cause | First checks | Mitigation |
|---------|--------------|--------------|------------|
| Latency spike | Downstream model / retrieve | p95 by stage, saturation | shed load, failover |
| Quality drop | Prompt/model/index change | diff versions, eval slice | rollback |
| Cost spike | loops / huge prompts | tokens/req, step counts | budget breaker |
| Security alert | injection / ACL | traces + retrieved IDs | kill switch |

Keep this table in your ops wiki; customize per system.


In [ ]:
# Operator helper — stage latency rollup
from statistics import mean

stages = {
    'gateway': [20, 25, 22],
    'retrieve': [80, 120, 95],
    'generate': [900, 1100, 980],
}
for k, v in stages.items():
    print(f'{k:10} mean={mean(v):.0f}ms max={max(v)}ms')
print('e2e~', sum(mean(v) for v in stages.values()), 'ms')


## Teaching notes (for study groups)

- Start with the comparison table; argue both sides for 5 minutes  
- Pair-program one demo cell with a real endpoint (placeholder keys)  
- Each person writes one failure case the suite must catch  
- End with a 60-second summary of when *not* to use the technique  


## Summary & Key Takeaways

- Agents close the loop with tools and environment feedback
- Minimal tool surface beats sprawling privileges
- Plan for loops, thrashing, and permission boundaries
